<a href="https://colab.research.google.com/github/697kiran/Breast-Cancer-Histology-Detection-ViT-AttentionFCNN-Radiomics/blob/main/vitrad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install torch torchvision transformers pandas numpy scikit-learn Pillow

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Check CUDA availability
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")

CUDA available: True
GPU Name: NVIDIA GeForce RTX 2050


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import os
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import warnings
from sklearn.metrics import roc_auc_score
from PIL import Image
from transformers import ViTForImageClassification, ViTImageProcessor
import time
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, precision_recall_curve

# Step 1: Define paths
base_path = 'C:/Users/user/Downloads/archive (8)/BreakHis - Breast Cancer Histopathological Database/dataset_cancer_v1/dataset_cancer_v1/classificacao_binaria'
radiomics_path = 'C:/Users/user/Downloads/radiomic_features_numeric.csv'

# Step 2: Load full radiomics data
radiomics_data = pd.read_csv(radiomics_path)
print("Columns in radiomic_features_numeric.csv:", radiomics_data.columns.tolist())
radiomics_data = radiomics_data.rename(columns={'filename': 'image_name'})
radiomics_data = radiomics_data.sample(frac=1, random_state=42).reset_index(drop=True)


# Optional: Shuffle the data for randomness
radiomics_data = radiomics_data.sample(frac=1, random_state=42).reset_index(drop=True)

# Verify and map 'class' column
print("Unique values in 'class' column:", radiomics_data['class'].unique())
if radiomics_data['class'].dtype == object:
    class_map = {'benign': 0, 'malignant': 1}
    radiomics_data['class'] = radiomics_data['class'].map(class_map)
    print("Mapped 'class' column to numeric values:", radiomics_data['class'].unique())

# Step 3: Split into train (75%), val (10%), test (15%)
train_radiomics, temp_radiomics = train_test_split(
    radiomics_data, test_size=0.25, stratify=radiomics_data['class'], random_state=42
)
val_radiomics, test_radiomics = train_test_split(
    temp_radiomics, test_size=0.6, stratify=temp_radiomics['class'], random_state=42
)

# Print dataset sizes per magnification
magnifications = ['40X', '100X', '200X', '400X']
for mag in magnifications:
    path_benign = os.path.join(base_path, mag, 'benign')
    path_malignant = os.path.join(base_path, mag, 'malignant')
    print(f"Benign ({mag}): {len(os.listdir(path_benign))} images")
    print(f"Malignant ({mag}): {len(os.listdir(path_malignant))} images")

# Step 4: Define image path generator
def get_image_path(row):
    subfolder = 'benign' if row['class'] == 0 else 'malignant'
    return os.path.join(base_path, row['magnification'], subfolder, row['image_name'])

# Step 5: Add image_path column
train_radiomics['image_path'] = train_radiomics.apply(get_image_path, axis=1)
val_radiomics['image_path'] = val_radiomics.apply(get_image_path, axis=1)
test_radiomics['image_path'] = test_radiomics.apply(get_image_path, axis=1)

# Step 6: Check for missing images
def check_missing_images(df):
    df['image_exists'] = df['image_path'].apply(os.path.exists)
    missing = df[~df['image_exists']]['image_name'].tolist()
    if missing:
        warnings.warn(f"Warning: The following images were not found: {missing}")
        return df[df['image_exists']]
    return df

train_radiomics = check_missing_images(train_radiomics)
val_radiomics = check_missing_images(val_radiomics)
test_radiomics = check_missing_images(test_radiomics)
print(f"Train set size after dropping missing images: {len(train_radiomics)}")
print(f"Validation set size after dropping missing images: {len(val_radiomics)}")
print(f"Test set size after dropping missing images: {len(test_radiomics)}")

# Step 7: Define patching function
def create_patches(image, patch_size=224):
    patches = []
    h, w = image.shape[:2]
    for i in range(0, h, patch_size):
        for j in range(0, w, patch_size):
            patch = image[i:i+patch_size, j:j+patch_size]
            if patch.shape[0] == patch_size and patch.shape[1] == patch_size:
                patches.append(patch)
    return patches

# Step 8: Stain normalization function
def stain_normalization(image):
    target_color = np.array([200, 165, 215])
    img = image.astype(np.float32)
    mean_color = np.mean(img, axis=(0, 1))
    img = img * (target_color / (mean_color + 1e-6))
    img = np.clip(img, 0, 255).astype(np.uint8)
    return img

# Step 9: Select top 10 texture features using Random Forest
texture_cols = [col for col in radiomics_data.columns if col.startswith(('original_glcm_', 'original_gldm_', 'original_glrlm_', 'original_glszm_', 'original_ngtdm_'))]
X_train_texture = train_radiomics[texture_cols]
y_train = train_radiomics['class']
X_val_texture = val_radiomics[texture_cols]
y_val = val_radiomics['class']
X_test_texture = test_radiomics[texture_cols]
y_test = test_radiomics['class']

scaler = StandardScaler()
X_train_texture_std = scaler.fit_transform(X_train_texture)
X_val_texture_std = scaler.transform(X_val_texture)
X_test_texture_std = scaler.transform(X_test_texture)

rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(X_train_texture_std, y_train)
feature_importances = pd.Series(rf_classifier.feature_importances_, index=texture_cols)
selected_features = feature_importances.nlargest(10).index.tolist()
print("Selected top 10 texture features:", selected_features)

# Select the top 10 texture features
train_texture_selected = train_radiomics[selected_features].values
val_texture_selected = val_radiomics[selected_features].values
test_texture_selected = test_radiomics[selected_features].values

# Step 10: Apply PCA to radiomic features (excluding selected texture features for ViT)
radiomic_cols = [col for col in radiomics_data.columns if col.startswith(('original_shape2D_', 'original_firstorder_'))]
scaler_pca = StandardScaler()
X_train_radiomic_std = scaler_pca.fit_transform(train_radiomics[radiomic_cols])
X_val_radiomic_std = scaler_pca.transform(val_radiomics[radiomic_cols])
X_test_radiomic_std = scaler_pca.transform(test_radiomics[radiomic_cols])

pca = PCA(n_components=40, random_state=42)
X_train_pca = pca.fit_transform(X_train_radiomic_std)
X_val_pca = pca.transform(X_val_radiomic_std)
X_test_pca = pca.transform(X_test_radiomic_std)
print(f"PCA transformed shape: {X_train_pca.shape}")

# Step 11: Define Custom ViT Model with Texture Features
class CustomViTWithTexture(ViTForImageClassification):
    def __init__(self, config, num_texture_features=10):
        super().__init__(config)
        self.texture_projection = nn.Linear(num_texture_features, config.hidden_size)
        self.combine_layer = nn.Linear(config.hidden_size * 2, config.hidden_size)

    def forward(self, pixel_values, texture_features, labels=None):
        vit_outputs = self.vit(pixel_values=pixel_values)
        cls_token = vit_outputs.last_hidden_state[:, 0, :]  # [batch_size, hidden_size]
        texture_emb = self.texture_projection(texture_features)  # [batch_size, hidden_size]
        combined = torch.cat((cls_token, texture_emb), dim=1)  # [batch_size, hidden_size * 2]
        combined = self.combine_layer(combined)  # [batch_size, hidden_size]
        logits = self.classifier(combined)
        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)
        return {'loss': loss, 'logits': logits} if loss is not None else {'logits': logits, 'features': combined}

# Step 12: Custom Dataset for ViT with Texture Features
class CustomDataset(Dataset):
    def __init__(self, image_paths, texture_features, labels, processor):
        self.image_paths = image_paths
        self.texture_features = texture_features
        self.labels = labels
        self.processor = processor

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        image = np.array(image)
        image = stain_normalization(image)
        patches = create_patches(image, patch_size=224)
        if len(patches) == 0:
            image = cv2.resize(image, (224, 224))
            patches = [image]
        image = Image.fromarray(patches[0])  # Use first patch
        image_inputs = self.processor(images=image, return_tensors="pt")
        texture = torch.tensor(self.texture_features[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image_inputs.pixel_values.squeeze(), texture, label

# Step 13: Initialize ViT and DataLoaders
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')
model = CustomViTWithTexture.from_pretrained('google/vit-base-patch16-224', num_texture_features=10)
model.to(device)

# Freeze ViT backbone
for param in model.vit.parameters():
    param.requires_grad = False

# Create datasets
train_dataset = CustomDataset(
    image_paths=train_radiomics['image_path'].values,
    texture_features=train_texture_selected,
    labels=train_radiomics['class'].values,
    processor=processor
)
val_dataset = CustomDataset(
    image_paths=val_radiomics['image_path'].values,
    texture_features=val_texture_selected,
    labels=val_radiomics['class'].values,
    processor=processor
)
test_dataset = CustomDataset(
    image_paths=test_radiomics['image_path'].values,
    texture_features=test_texture_selected,
    labels=test_radiomics['class'].values,
    processor=processor
)

# Create DataLoaders
train_loader_vit = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader_vit = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader_vit = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Step 14: Extract ViT Features with Texture Integration
def extract_vit_features(loader, model, device):
    model.eval()
    features = []
    with torch.no_grad():
        for pixel_values, texture_features, _ in loader:
            pixel_values, texture_features = pixel_values.to(device), texture_features.to(device)
            outputs = model(pixel_values=pixel_values, texture_features=texture_features)
            features.append(outputs['features'].cpu().numpy())
    return np.vstack(features)

train_vit_features = extract_vit_features(train_loader_vit, model, device)
val_vit_features = extract_vit_features(val_loader_vit, model, device)
test_vit_features = extract_vit_features(test_loader_vit, model, device)

# Step 15: Standardize ViT Features
scaler_vit = StandardScaler()
train_vit_features_std = scaler_vit.fit_transform(train_vit_features)
val_vit_features_std = scaler_vit.transform(val_vit_features)
test_vit_features_std = scaler_vit.transform(test_vit_features)

# Step 16: Combine ViT Features with PCA Features
combined_train = np.hstack((train_vit_features_std, X_train_pca))
combined_val = np.hstack((val_vit_features_std, X_val_pca))
combined_test = np.hstack((test_vit_features_std, X_test_pca))

# To tensor (features)
X_train = torch.tensor(combined_train, dtype=torch.float32)
X_val = torch.tensor(combined_val, dtype=torch.float32)
X_test = torch.tensor(combined_test, dtype=torch.float32)

# Convert labels to tensor
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.long)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

# Datasets & Loaders for FCNN
train_ds = TensorDataset(X_train, y_train_tensor)
val_ds = TensorDataset(X_val, y_val_tensor)
test_ds = TensorDataset(X_test, y_test_tensor)
train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=512, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=512, shuffle=False)

# Step 17: Model Definition
class AttentionFCNN(nn.Module):
    def __init__(self, input_dim=808, hidden_dims=[512, 256], num_classes=2, dropout_rate=0.6):
        super(AttentionFCNN, self).__init__()
        layers = []
        prev_dim = input_dim
        for dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate)
            ])
            prev_dim = dim
        layers.append(nn.Linear(prev_dim, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

# Step 18: AUC-safe Evaluation Helper
def evaluate(model, loader, device):
    model.eval()
    all_probs, all_preds, all_labels = [], [], []
    with torch.no_grad():
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            logits = model(Xb)
            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = torch.argmax(logits, dim=1)
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(yb.cpu().numpy())
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = float('nan')
    return acc, auc, all_probs, all_preds, all_labels

# Step 19: Training Loop with Metric Storage
model_fnn = AttentionFCNN(input_dim=808).to(device)  # 768 (ViT) + 40 (PCA)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model_fnn.parameters(), lr=1e-3)
num_epochs = 10

train_losses = []
train_accs = []
train_aucs = []
val_accs = []
val_aucs = []

for epoch in range(1, num_epochs+1):
    model_fnn.train()
    running_loss = 0.0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model_fnn(Xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * Xb.size(0)
    epoch_loss = running_loss / len(train_ds)
    train_losses.append(epoch_loss)
    train_acc, train_auc, _, _, _ = evaluate(model_fnn, train_loader, device)
    val_acc, val_auc, _, _, _ = evaluate(model_fnn, val_loader, device)
    train_accs.append(train_acc)
    train_aucs.append(train_auc)
    val_accs.append(val_acc)
    val_aucs.append(val_auc)
    print(f"Epoch {epoch:2d}  "
          f"Loss: {epoch_loss:.4f}  "
          f"Train Acc: {train_acc*100:5.2f}%  "
          f"Train AUC: {train_auc:5.4f}  "
          f"Val Acc: {val_acc*100:5.2f}%  "
          f"Val AUC: {val_auc:5.4f}")

# Evaluate on test set
test_acc, test_auc, test_probs, test_preds, test_labels = evaluate(model_fnn, test_loader, device)
print(f"\nFinal Test Metrics:")
print(f"Test Acc: {test_acc*100:5.2f}%  Test AUC: {test_auc:5.4f}")

# Save the trained model
torch.save(model_fnn.state_dict(), 'model.pth')

# Step 20: Plotting Results
sns.set_theme(style="darkgrid")
sns.set_context("notebook", font_scale=1.2)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

# Loss Curve
ax1.plot(range(1, num_epochs+1), train_losses, label='Train Loss', marker='o')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('Training Loss Curve')
ax1.legend()
ax1.grid(True)

# Accuracy Curve
ax2.plot(range(1, num_epochs+1), train_accs, label='Train Accuracy', marker='o')
ax2.plot(range(1, num_epochs+1), val_accs, label='Validation Accuracy', marker='s')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy Curves')
ax2.legend()
ax2.grid(True)

# AUC Curve
ax3.plot(range(1, num_epochs+1), train_aucs, label='Train AUC', marker='o')
ax3.plot(range(1, num_epochs+1), val_aucs, label='Validation AUC', marker='s')
ax3.set_xlabel('Epoch')
ax3.set_ylabel('AUC-ROC')
ax3.set_title('AUC-ROC Curves')
ax3.legend()
ax3.grid(True)

plt.tight_layout()
plt.show()

# Confusion Matrix
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benign', 'Malignant'],
            yticklabels=['Benign', 'Malignant'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix (Test Set)')
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(test_labels, test_probs)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {test_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve (Test Set)')
plt.legend()
plt.grid(True)
plt.show()

# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(test_labels, test_probs)
plt.figure(figsize=(6, 5))
plt.plot(recall, precision, label='Precision-Recall Curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve (Test Set)')
plt.legend()
plt.grid(True)
plt.show()

# Prediction Distribution
plt.figure(figsize=(8, 5))
sns.histplot(data={'Benign': np.array(test_probs)[np.array(test_labels) == 0],
                   'Malignant': np.array(test_probs)[np.array(test_labels) == 1]},
             bins=30, stat='density', common_norm=False, alpha=0.5)
plt.xlabel('Predicted Probability (Malignant)')
plt.ylabel('Density')
plt.title('Prediction Distribution (Test Set)')
plt.legend(['Benign', 'Malignant'])
plt.grid(True)
plt.show()
```